In [1]:
# make sure that this is the expected virtual env
!which python

/Users/awlego/Repositories/lc0/env3.10/bin/python


In [2]:
# /opt/homebrew/bin/lc0
!pip install numpy
!pip install scikit-learn
!pip install tensorflow
!pip install gzip
!pip install tf2onnx
!pip install onnx
!pip install onnxruntime


[notice] A new release of pip available: 22.2.2 -> 23.3.1
[notice] To update, run: pip install --upgrade pip
  Using cached scikit_learn-1.3.2-cp310-cp310-macosx_12_0_arm64.whl (9.5 MB)
  Using cached joblib-1.3.2-py3-none-any.whl (302 kB)
  Using cached scipy-1.11.3-cp310-cp310-macosx_12_0_arm64.whl (29.8 MB)
  Using cached threadpoolctl-3.2.0-py3-none-any.whl (15 kB)

[notice] A new release of pip available: 22.2.2 -> 23.3.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip available: 22.2.2 -> 23.3.1
[notice] To update, run: pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement gzip (from versions: none)
ERROR: No matching distribution found for gzip

[notice] A new release of pip available: 22.2.2 -> 23.3.1
[notice] To update, run: pip install --upgrade pip
  Using cached tf2onnx-1.15.1-py3-none-any.whl (454 kB)
  Using cached protobuf-3.20.3-py2.py3-none-any.whl (162 kB)
  Attempting uninstall: protobuf
    Found 

In [11]:
import numpy as np
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
from sklearn.decomposition import NMF
import gzip
import onnx
import onnxruntime
from lczero.backends import Weights, Backend, GameState

In [4]:
Lc0_model_path = "models/t2-768x15x24h-swa-5230000.pb"
Lc0_model_path2 = "/opt/homebrew/Cellar/lc0/0.30.0/libexec/42850.pb.gz"
onnx_path = "/opt/homebrew/Cellar/lc0/0.30.0/libexec/42850.onnx"

In [5]:
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)


In [6]:
import onnxruntime as ort

sess_options = ort.SessionOptions()

# You can set various session options here if needed
# For example, to enable GPU: sess_options.device.id = "GPU"

sess = ort.InferenceSession(onnx_path, sess_options)

In [70]:
input_name = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name

# Here's a dummy input for demonstration. Make sure you replace it with your actual data
dummy_input_data = np.random.randn(1, 112, 8, 8).astype(np.float32)
print(dummy_input_data.shape)

predictions = sess.run([output_name], {input_name: dummy_input_data})
print(predictions)


(1, 112, 8, 8)
[array([[  806.534  ,    81.16784,   683.0105 , ..., -3371.7163 ,
        -1966.5831 , -2539.8945 ]], dtype=float32)]


In [8]:
# okay I can look at my onnx model in netron.app and see it looks right -- I see 15 blocks.

In [9]:
new_board_fen = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"
e4e5_board_fen = "rnbqkbnr/pppp1ppp/8/4p3/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2"

In [142]:
w = Weights(Lc0_model_path2)
b = Backend(weights=w)
g = GameState(new_board_fen)
print(g.as_string())
g2 = GameState(e4e5_board_fen)
print(g2.as_string())

rnbqkbnr
pppppppp
........
........
........
........
PPPPPPPP
RNBQKBNR KQkq[AHah] (from white's eyes) Hash: 2326312736498822375

rnbqkbnr
pppp.ppp
........
....p...
....P...
........
PPPP.PPP
RNBQKBNR KQkq[AHah] (from white's eyes) Hash: 8572518730975593504



Creating backend [metal]...
Initialized metal backend on device Apple M1 Max


In [31]:
dir(g2)

['__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'as_input',
 'as_string',
 'moves',
 'policy_indices']

#todo:
1) encode a board state from a FEN into the format the the model takes
2) write something to translate the output into top 3 moves
3) run the model on a new game and an e4e5 game to validate that the predictions match the chess engine version I have
4) do the nmf stuff

In [35]:
dir(g2.as_input(b))

['__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'clone',
 'mask',
 'set_mask',
 'set_val',
 'val']

In [276]:
print(bin(g2.as_input(b).mask(0)))
dummy_input_data = np.random.randn(1, 112, 8, 8).astype(np.bool_)
dummy_input_data[0, 1]



0b10000000000001110111100000000


array([[ True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True]])

In [168]:
bin(g2.as_input(b).mask(0))
print(len(bin(g2.as_input(b).mask(0))[2:]))

29


In [172]:
number = '0b10000000000001110111100000000'
len(number[2:])

29

In [64]:
112*8*8


7168

how can I map 29 bits x 112 channels onto 112 8x8 that the input board looks like? Is it compressed?

In [326]:
def print_binary_as_8x8(number):
    # Convert the number to a binary string, removing the '0b' prefix, and pad with leading zeros
    binary_string = format(number, '064b')
    # Split the binary string into 8-character chunks
    rows = [binary_string[i:i+8] for i in range(0, len(binary_string), 8)]
    # flip the bits in the row because the first bit the bottom left not bottom right
    rows = [row[::-1] for row in rows]
    # Print each row to form the 8x8 grid
    for row in rows:
        print(' '.join(row))

# Example usage:
print_binary_as_8x8(255)  # This


0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
1 1 1 1 1 1 1 1


// Represents a board as an array of 64 bits.  
// Bit enumeration goes from bottom to top, from left to right:  
// Square a1 is bit 0, square a8 is bit 7, square b1 is bit 8.  
class BitBoard {  

In [329]:
# print(dir(g2))
print(g2.as_string())

# if (board.castlings().we_can_000()) result[kAuxPlaneBase + 0].SetAll();
# should be all 1s if we can 000 castle
print("this should be all 1s if we can 000 castle")
print(bin(g2.as_input(b).mask(104)))
print("")

# result[base + 0].mask = (board.ours() & board.pawns()).as_int();
kPlanesPerBoard = 13
print("This should be our pawns")
print_binary_as_8x8(g2.as_input(b).mask(0))
print("This should be our knights")
print_binary_as_8x8(g2.as_input(b).mask(1))


rnbqkbnr
pppp.ppp
........
....p...
....P...
........
PPPP.PPP
RNBQKBNR KQkq[AHah] (from white's eyes) Hash: 8572518730975593504

this should be all 1s if we can 000 castle
0b1111111111111111111111111111111111111111111111111111111111111111

This should be our pawns
0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
0 0 0 0 1 0 0 0
0 0 0 0 0 0 0 0
1 1 1 1 0 1 1 1
0 0 0 0 0 0 0 0
This should be our knights
0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
0 1 0 0 0 0 1 0


In [332]:
print_binary_as_8x8(g2.as_input(b).mask(26))


0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0
0 0 0 0 1 0 0 0
0 0 0 0 0 0 0 0
1 1 1 1 0 1 1 1
0 0 0 0 0 0 0 0


In [306]:
num = 10
print(format(num, '064b'))

0000000000000000000000000000000000000000000000000000000000001010


In [343]:
e4e5_board_fen = "rnbqkbnr/pppp1ppp/8/4p3/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2"
w = Weights(Lc0_model_path2)
b = Backend(weights=w)
g2 = GameState(e4e5_board_fen)

def mask_to_plane(mask):
    # Convert to binary string, strip off the '0b' prefix, and pad to 64 bits
    binary_string = bin(mask)[2:].zfill(64)
    rows = [binary_string[i:i+8] for i in range(0, len(binary_string), 8)]
    rows = [row[::-1] for row in rows]

    plane = np.array([[float(bit) for bit in row] for row in rows])
    return plane

# Create an empty array of shape (112, 8, 8)
board_planes = np.zeros((112, 8, 8), dtype=np.float32)

for i in range(112):
    board_planes[i] = mask_to_plane(g2.as_input(b).mask(i))
# Add a batch dimension
input_data = board_planes[np.newaxis, :]

Creating backend [metal]...
Initialized metal backend on device Apple M1 Max


In [349]:
input_data[0][0]

array([[0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 1., 1., 1., 0., 1., 1., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)

In [344]:
predictions = sess.run([output_name], {input_name: input_data})
print(len(predictions[0][0]))
# 1858 is the the /output/policy shape (batch x 1858) as seen in netron.app
print(np.argsort(predictions[0].flatten())[::-1])


1858
[1674 1495  104 ... 1710 1648 1709]


In [340]:
def p_softmax(indicies, p_, p_raw_func):
    """
    Convert raw logits to softmax probabilities.

    Args:
        indicies (list[int]): List of indices to consider.
        p_raw_func (callable): A function that accepts a list of indices and returns the corresponding raw logits.

    Returns:
        list[float]: Softmax probabilities.
    """
    # Get raw logits for the provided indices
    p_vals = np.array(p_raw_func(indicies, p_))
    
    # Calculate the maximum value for numerical stability
    max_p = np.max(p_vals)
    
    # Calculate exponential values
    p_vals = np.exp(p_vals - max_p)
    
    # Normalize to get probabilities
    total = np.sum(p_vals)
    if total > 0:
        p_vals /= total
    
    return p_vals.tolist()

def p_raw(indicies, p_):
    """
    Extract raw logits from the provided p_ array based on given indices.

    Args:
        indicies (list[int]): List of indices to consider.
        p_ (list[float]): Array containing raw logits or scores.

    Returns:
        list[float]: Extracted raw logits or scores.
    """
    if any(idx < 0 or idx > 1857 for idx in indicies):
        raise ValueError("Policy index must be between 0 and 1857.")
    
    return [p_[idx] for idx in indicies]


In [341]:
# raw_logits = p_raw(g.policy_indices(), predictions[0].flatten())
# predictions at 0 since our predictions are batched and we have a batch of 1 here
list(zip(g2.moves(), p_softmax(g2.policy_indices(), predictions[0].flatten(), p_raw)))


[('b1a3', 0.038827307522296906),
 ('b1c3', 0.036406680941581726),
 ('d1e2', 0.036989811807870865),
 ('d1f3', 0.03745288401842117),
 ('d1g4', 0.03692179173231125),
 ('d1h5', 0.03598584979772568),
 ('e1e2', 0.03628568351268768),
 ('f1e2', 0.034554459154605865),
 ('f1d3', 0.039538703858852386),
 ('f1c4', 0.037077732384204865),
 ('f1b5', 0.03672941029071808),
 ('f1a6', 0.03707624971866608),
 ('g1e2', 0.035507574677467346),
 ('g1f3', 0.044104695320129395),
 ('g1h3', 0.03570499271154404),
 ('a2a3', 0.036415211856365204),
 ('a2a4', 0.03552195802330971),
 ('b2b3', 0.03777836635708809),
 ('b2b4', 0.03431088104844093),
 ('c2c3', 0.029396358877420425),
 ('c2c4', 0.02988540008664131),
 ('d2d3', 0.02588292397558689),
 ('d2d4', 0.03454113006591797),
 ('f2f3', 0.027828771620988846),
 ('f2f4', 0.025179974734783173),
 ('g2g3', 0.031716521829366684),
 ('g2g4', 0.030104247853159904),
 ('h2h3', 0.032940857112407684),
 ('h2h4', 0.029333652928471565)]

In [266]:
g2.policy_indices()
print(g2.as_string())

rnbqkbnr
pppp.ppp
........
....p...
....P...
........
PPPP.PPP
RNBQKBNR KQkq[AHah] (from white's eyes) Hash: 8572518730975593504



# Let's calculate the true Lc0 output so I can validate that my representations are working correctly

In [342]:
w = Weights(Lc0_model_path2)
b = Backend(weights=w)
# g2 = GameState(e4e5_board_fen)
dummy_input = GameState(e4e5_board_fen).as_input(b)
input = g2.as_input(b)
output, _ = b.evaluate(input, dummy_input)
dir(output)
list(zip(g2.moves(), output.p_softmax(*g2.policy_indices())))

Creating backend [metal]...
Initialized metal backend on device Apple M1 Max


[('b1a3', 0.0017741298070177436),
 ('b1c3', 0.17404316365718842),
 ('d1e2', 0.001867432612925768),
 ('d1f3', 0.002248866017907858),
 ('d1g4', 0.0018720037769526243),
 ('d1h5', 0.0020443403627723455),
 ('e1e2', 0.0014642218593508005),
 ('f1e2', 0.01243040431290865),
 ('f1d3', 0.004636666271835566),
 ('f1c4', 0.04721570760011673),
 ('f1b5', 0.003496352816000581),
 ('f1a6', 0.0017000455409288406),
 ('g1e2', 0.004145870916545391),
 ('g1f3', 0.6667669415473938),
 ('g1h3', 0.0018152031116187572),
 ('a2a3', 0.010607978329062462),
 ('a2a4', 0.006029128096997738),
 ('b2b3', 0.0021638551261276007),
 ('b2b4', 0.0017414273461326957),
 ('c2c3', 0.004176905378699303),
 ('c2c4', 0.003008705098181963),
 ('d2d3', 0.013671309687197208),
 ('d2d4', 0.011745461262762547),
 ('f2f3', 0.0018560135504230857),
 ('f2f4', 0.002120622666552663),
 ('g2g3', 0.0027426090091466904),
 ('g2g4', 0.001520868157967925),
 ('h2h3', 0.009203264489769936),
 ('h2h4', 0.0018903155578300357)]

In [274]:
bin(input.mask(0))

'0b10000000000001110111100000000'

NameError: name 'a' is not defined

In [259]:
def perform_nmf(activations, num_inputs=10**4, num_factors=36, random_state=None):
    """
    :param activations: a list of activations for multiple inputs, each of shape (H, W, C)
    :param num_inputs: number of random inputs to be selected for factorization
    :param num_factors: number of factors (K) to use in NMF
    :param random_state: seed for reproducibility
    :return: F, Omega_all, approximations for each input activation
    """
    
    # Reshape each activation to create z^l_{hat}
    # original gpt4 output:
    # reshaped_activations = [a.reshape(-1, a.shape[-1]) for a in activations]
    # what I think it should be so that I "We treat each plane as a vector" reshaping 8x8 into 64
    reshaped_activations = [a.flatten() for a in dummy_matrix]

    # Randomly select `num_inputs` from reshaped activations and stack them
    random_selected = np.vstack(np.random.choice(reshaped_activations, num_inputs, replace=False))
    
    # Perform NMF
    nmf = NMF(n_components=num_factors, init='random', random_state=random_state)
    Omega_all = nmf.fit_transform(random_selected)
    F = nmf.components_
    
    # For each input activation, find its compressed representation (weights) in terms of the global factors F
    approximations = [nmf.transform(ra) for ra in reshaped_activations]
    
    return F, Omega_all, approximations

# Example usage:
# Given: activations = [activation1, activation2, ...] where each activation is a numpy array of shape (H, W, C)
# 8x8x112
dummy_matrix = np.random.randint(low=0, high=2, size=(112, 8, 8))
for a in dummy_matrix:
    print(a.shape)
    break
reshaped_activations = [a.flatten() for a in dummy_matrix]

print(len(reshaped_activations[1]))

# activations 
F, Omega_all, approximations = perform_nmf(dummy_matrix, random_state=0)


(8, 8)
64


ValueError: a must be 1-dimensional

In [218]:
import numpy as np
X = np.random.randint(low=0, high=10, size=(8, 8))
from sklearn.decomposition import NMF
model = NMF(n_components=36, init='random', random_state=0)
W = model.fit_transform(X)
H = model.components_

In [221]:
H.shape

(36, 8)

In [220]:
W.shape

(8, 36)

In [223]:
print(X)
np.matmul(W, H)

[[7 2 3 4 2 4 9 8]
 [9 0 7 5 2 9 6 6]
 [7 2 0 5 2 2 2 9]
 [1 3 0 9 4 4 6 9]
 [0 4 2 8 1 1 5 8]
 [4 5 9 0 0 5 0 6]
 [7 8 7 4 6 8 4 3]
 [9 6 4 5 8 1 6 4]]


array([[7.00021627e+00, 2.00000207e+00, 3.00030750e+00, 4.00000997e+00,
        1.99999593e+00, 4.00001820e+00, 9.00000014e+00, 7.99999506e+00],
       [9.00002355e+00, 5.14921445e-05, 6.99997644e+00, 5.00000038e+00,
        1.99999860e+00, 9.00001176e+00, 6.00000819e+00, 6.00000154e+00],
       [7.00001723e+00, 1.99999865e+00, 2.76556998e-02, 4.99999467e+00,
        1.99999918e+00, 1.99999781e+00, 2.00000850e+00, 9.00000133e+00],
       [9.99986990e-01, 3.00000514e+00, 2.68943824e-02, 9.00000342e+00,
        4.00000276e+00, 3.99995818e+00, 5.99998499e+00, 8.99999476e+00],
       [3.64040903e-03, 3.99999718e+00, 2.00004918e+00, 7.99999812e+00,
        9.99998482e-01, 1.00002294e+00, 5.00000825e+00, 8.00000288e+00],
       [3.99975168e+00, 5.00000155e+00, 8.99979238e+00, 6.27246219e-04,
        3.67445920e-04, 4.99997989e+00, 5.34779476e-04, 5.99999703e+00],
       [6.99998764e+00, 8.00000095e+00, 7.00039495e+00, 4.00000547e+00,
        6.00000737e+00, 8.00000022e+00, 3.99997674e+00, 2.